In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

# 1. Load the generated features from S19
df_features = pd.read_csv('matminer_magpie_features.csv')

initial_count = df_features.shape[1]
print(f"Initial feature count: {initial_count}")

# 2. Variance Filter
numeric_df = df_features.select_dtypes(include=[np.number])

selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_df)

# Get columns that passed the variance test
cols_to_keep = numeric_df.columns[selector.get_support()]
df_var_filtered = numeric_df[cols_to_keep]

var_count = df_var_filtered.shape[1]

print(f"Features after Variance Filter: {var_count} "
      f"(Dropped {initial_count - var_count})")

# 3. Correlation Filter
corr_matrix = df_var_filtered.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

df_final = df_var_filtered.drop(columns=to_drop)

final_count = df_final.shape[1]

print(f"Features after Correlation Filter: {final_count} "
      f"(Dropped {len(to_drop)})")

# 4. Save the filtered dataset
df_final.to_csv('filtered_features_s20.csv', index=False)

print("✅ First round of feature selection complete.")

Initial feature count: 132
Features after Variance Filter: 126 (Dropped 6)
Features after Correlation Filter: 96 (Dropped 30)
✅ First round of feature selection complete.
